In [9]:
import pandas as pd
import os
import zipfile
import shutil

# 경로 설정
zip_path = "drive/MyDrive/2025인공지능6팀/2. 형태 주석 말뭉치.zip"  # 모든 sample_xxx.xls가 들어있는 zip
meta_path = "drive/MyDrive/2025인공지능6팀/2015~2023 학습자 말뭉치 표본 정보_20241031(최종공개용).xlsx"  # 표본 정보 파일
output_csv = "corpus_text_label.csv"
extract_dir = "./unzipped_samples"

# 1. samples.zip 압축 해제 (필요시 덮어쓰기)
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# 2. 말뭉치 표본정보 읽기 및 필터링
meta_df = pd.read_excel(meta_path)
meta_df = meta_df[meta_df['작문 유형'] == "1 인 발화"]
meta_df = meta_df[['표본ID', '한국어 등급']].dropna()

# 3. 결과 생성
results = []

for _, row in meta_df.iterrows():
    sample_id = str(row['표본ID']).zfill(3)  # 예: 001
    label_str = str(row['한국어 등급'])
    if not label_str or not label_str[0].isdigit():
      continue  # 숫자로 시작하지 않으면 무시
    label = int(label_str[0])
    sample_path = os.path.join(extract_dir, f"sample_{sample_id}.xls")

    if not os.path.exists(sample_path):
        continue

    try:
        df = pd.read_excel(sample_path)
    except Exception as e:
        print(f"{sample_id} 읽기 오류: {e}")
        continue

    # 문장 기준으로 묶고 형태소를 조합
    sentence_list = []
    grouped = df.groupby(['문장'])
    for _, group in grouped:
        word_list = []
        for _, row in group.iterrows():
            if pd.isna(row['원 형태소']) or pd.isna(row['형태 주석']):
                continue  # NaN인 경우 건너뜀
            word = str(row['원 형태소']) + '/' + str(row['형태 주석'])
            word_list.append(word)
        sentence = ' '.join(word_list)
        sentence_list.append(sentence)

    full_text = ' '.join(sentence_list)
    results.append({'text': full_text, 'label': label})

# 4. CSV로 저장
out_df = pd.DataFrame(results)
out_df.to_csv(output_csv, index=False)
print(f"CSV 저장 완료: {output_csv} ({len(out_df)} rows)")

CSV 저장 완료: corpus_text_label.csv (947 rows)
